In [1]:
import sys
import os
import json

# Add the parent directory to the system path to access 'utils'
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(parent_dir)
sys.path.append(parent_dir)

In [2]:
from utils.latex_to_json import *
tex_dir = os.path.join(parent_dir, "gts")
json_dir = os.path.join(parent_dir, "jsons")


In [3]:

# # Loop over all files in the directory
# for file in os.listdir(tex_dir):
#     if file.endswith(".tex"):
#         # Try-catch block to handle any errors
#         try:
#             print(f"Processing {file}")
#             # Convert the tex file to json
#             out_json = tex_file_to_json(os.path.join(tex_dir, file))
#             # with open(os.path.join(json_dir, file.replace(".tex", ".json")), "w") as f:
#             #     json.dump(out_json, f, indent=4)

#         except Exception a
# s e:
#             print(f"Error processing {file}: {e}")
#             break


In [4]:
dictt = tex_file_to_json(os.path.join(tex_dir, "c_20231231_cut.txt"))

##save as json
with open(os.path.join(json_dir, "c_20231231_cut.json"), "w") as f:
    json.dump(dictt, f, indent=4)



In [1]:
from TexSoup import TexSoup as TS
import networkx as nx
import matplotlib.pyplot as plt

import json

tex_dir = "t.tex"

In [3]:
with open(tex_dir) as f: data = f.read()
soup = TS(data)

In [1]:
HIERARCHY = ['document', 'section', 'subsection', 'subsubsection', 'paragraph', 'subparagraph']

LEAF_NODES = ['itemize', 'table', 'enumerate']
MAX_TEXT_LENGTH = 1500000

from TexSoup import TexSoup as TS

def tex_soup_to_json(tex_content):
    doc_index = 0
    content_count = len(tex_content.contents)
    
    for i in range(content_count):
        if not isinstance(tex_content.contents[i], str) and tex_content.contents[i].name == 'document':
            doc_index = i
            break

    document_content = tex_content.contents[doc_index]
    node_id = 0
    node_stack = [{'id': node_id, 'name': 'document', 'level': 0, 'type': 'document', 'children': []}]
    node_id += 1

    for element in document_content:
        if isinstance(element, str):
            text_length = len(element)
            truncated_text = element[:min(MAX_TEXT_LENGTH, text_length)]
            node_stack[-1]['children'].append(truncated_text)

        elif element.name in HIERARCHY:
            element_depth = HIERARCHY.index(element.name)
            
            while element_depth <= HIERARCHY.index(node_stack[-1]['type']) and len(node_stack) > 1:
                node_stack.pop()

            if element_depth - 1 == HIERARCHY.index(node_stack[-1]['type']):
                new_node = {
                    'id': node_id,
                    'name': element.contents[0],
                    'type': element.name,
                    'children': [],
                    'level': node_stack[-1]['level'] + 1
                }
                node_stack[-1]['children'].append(new_node)
                node_stack.append(new_node)
                node_id += 1
            elif len(node_stack) > 1:
                raise Exception("Document is not structured with proper hierarchy")
        
        elif element.name in LEAF_NODES:

            children = []
            if element.name != 'table':
                children = [item.contents[0] for item in element.contents]


            leaf_node = {
                'id': node_id,
                'name': element.name,
                'level': node_stack[-1]['level'] + 1,
                'type': 'leaf',
                'children': children
            }
            node_stack[-1]['children'].append(leaf_node)
            node_id += 1


    return node_stack[0]

def tex_file_to_json(file_path):
    with open(file_path) as file:
        tex_data = file.read()
    tex_soup = TS(tex_data)
    return tex_soup_to_json(tex_soup)

In [ ]:
dictt = tex_file_to_json(tex_dir)
draw_dict(dictt)

In [52]:
# import  json
# j = to_dictionary(soup)
# print(f"Instance: Unknown ({type(soup.section)})")
# children



# json_data= {"doc" : j} ## j is list

# # Output the result in JSON format
# json_output = json.dumps(json_data, indent=2)
# print(json_output)
# print(str(soup.secion))
# print(soup.contents[0].contents[0])

# print(soup.contents[1].contents[0].name)
# print(soup.contents[1].contents[1])
# print(soup.contents[1].contents[2])
# print(soup.contents[1].contents[3])

# dictt = to_json(soup)




In [87]:
# pip install networkx matplotlib

In [ ]:
draw_tree(soup)

In [ ]:
toc.print()

In [9]:
def iterate_tree(node, level=0):
    # Print current element with indentation based on level
    print("\t" * level + str(node))
    
    # Recursively iterate over child elements if any
    for child in node.children:
        iterate_tree(child, level + 1)

In [10]:
import json
from TexSoup import TexSoup

# Sample LaTeX content
latex_code = r"""
\documentclass[a4paper]{article}
\begin{document}

\section{Chikin Tales}

\subsection{Chikin Fly}

Chickens don't fly. They do only the following:

\begin{itemize}
\item waddle
\item plop
\end{itemize}

\section{Chikin Scream}

\subsection{Plopping}

Plopping involves three steps:

\begin{enumerate}
\item squawk
\item plop
\item repeat, unless ordered to squat
\end{enumerate}

\subsection{I Scream}

\end{document}
"""

# Parse the LaTeX document
soup = TexSoup(latex_code)

# Function to recursively convert TexSoup nodes to a JSON-friendly dictionary



In [ ]:
# Initializ
# e TexSoap

from TexSoup import TexSoup
ts = TexSoap()

In [ ]:
# Convert to JSON
json_output = ts.convert(latex_code, output="json")

# Print the JSON output
print(json_output)

In [18]:
def to_dictionary(tex_tree):
    str_tree = []
    for i in tex_tree:
        if isinstance(i, list):
            str_tree.append(i)
        elif isinstance(i, TexSoup.TexEnv):
            str_tree.append(
                {
                    i.name: [
                        {"begin": i.begin + str(i.args)},
                        to_dictionary(i.all),
                        {"end": i.end},
                    ]
                }
            )
        elif isinstance(i, TexSoup.TexCmd):
            str_tree.append({i.name: "\\" + i.name + str(i.args)})
        elif isinstance(i, TexSoup.TexText):
            str_tree.append(str(i.text))
        elif isinstance(i, TexSoup.TexGroup):
            str_tree.append(["{", to_dictionary(TexSoup.TexSoup(i.value).expr.all), "}"])
        else:
            str_tree.append(str(i))

    return str_tree

In [ ]:
from TexSoup import TexSoup

# Initialize TexSoap
ts = TexSoup(latex_code)


to_dictionary(ts)